In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import glob

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 32
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## IDEA GSoD Pipeline

**Source:** International IDEA Global State of Democracy Indices
**Access:** Manual CSV download (auto-detects version in Downloads)
**Download instructions:** See `docs/instructions_data_maintenance.md` — IDEA_PARTIP section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Participation subcomponents | Political participation | Primary tier 2 |
| Representation, Rights, Rule of Law, Participation categories | Multiple | Cross-check |

In [2]:
import pandas as pd
import os
import glob
from datetime import datetime

# Auto-detect latest GSoD CSV in Downloads — no hardcoded version
# Pattern: gsod_indices_v{N}.csv — pick highest version number
gsod_pattern = os.path.join(DOWNLOADS_DIR, "gsod_indices_v*.csv")
gsod_files = glob.glob(gsod_pattern)

if not gsod_files:
    print(f"No GSoD file found in {DOWNLOADS_DIR}")
    print("Download from https://www.idea.int/democracytracker/gsod-indices")
else:
    # Sort by version number extracted from filename
    def extract_version(path):
        import re
        m = re.search(r'gsod_indices_v(\d+)', path)
        return int(m.group(1)) if m else 0
    
    latest_file = max(gsod_files, key=extract_version)
    gsod_version = extract_version(latest_file)
    print(f"Loading: {os.path.basename(latest_file)} (version {gsod_version})")
    
    gsod_raw = pd.read_csv(latest_file)
    print(f"\nShape: {gsod_raw.shape}")
    print(f"Columns: {list(gsod_raw.columns[:20])}")

Loading: gsod_indices_v10.csv (version 10)

Shape: (9863, 248)
Columns: ['Unnamed: 0', 'COWcode', 'year', 'country_year', 'iso3c', 'country_name', 'country_name_full', 'region_name', 'region', 'subregion_name', 'subregion', 'representation_est', 'representation_u', 'representation_l', 'representation_rank', 'cred_elect_est', 'cred_elect_l', 'cred_elect_u', 'inclu_suff_est', 'free_parties_est']


In [3]:
# Identify the main GSoD category estimate columns (_est suffix)
est_cols = [c for c in gsod_raw.columns if c.endswith('_est')]
print(f"Estimate columns ({len(est_cols)}):")
for col in est_cols[:40]:
    print(f"  {col}")

Estimate columns (30):
  representation_est
  cred_elect_est
  inclu_suff_est
  free_parties_est
  elected_gov_est
  effect_parl_est
  local_dem_est
  rights_est
  access_just_est
  civil_lib_est
  free_express_est
  free_press_est
  free_assoc_assem_est
  free_relig_est
  free_move_est
  basic_welf_est
  pol_equal_est
  soc_grp_equal_est
  econ_equal_est
  gender_equal_est
  rule_law_est
  jud_ind_est
  abs_corrupt_est
  predict_enf_est
  pers_integ_sec_est
  participation_est
  civil_soc_est
  civic_engage_est
  elect_part_est
  direct_dem_est


In [4]:
# Select framework-relevant GSoD category estimates
# Maps to concepts: political participation, civil society, electoral, rule of law cross-checks

KEEP_COLS = {
    # Identifiers
    'iso3c':         'country_code',
    'country_name':  'country_name',
    'COWcode':       'cow_code',
    'year':          'year',

    # Participation category — Primary tier 2 for Political participation (Concept 21)
    'participation_est':  'idea_participation',
    'civil_soc_est':      'idea_civil_society',
    'civic_engage_est':   'idea_civic_engagement',
    'elect_part_est':     'idea_electoral_participation',
    'direct_dem_est':     'idea_direct_democracy',
    'local_dem_est':      'idea_local_democracy',

    # Representation — Electoral process cross-check (Concept 20)
    'representation_est': 'idea_representation',
    'cred_elect_est':     'idea_credible_elections',
    'free_parties_est':   'idea_free_political_parties',
    'effect_parl_est':    'idea_effective_parliament',

    # Rights — Civil liberties cross-check
    'free_assoc_assem_est': 'idea_free_association_assembly',

    # Rule of Law — cross-check
    'jud_ind_est':        'idea_judicial_independence',
}

# Filter and rename
gsod = gsod_raw[[c for c in KEEP_COLS.keys() if c in gsod_raw.columns]].copy()
gsod = gsod.rename(columns={k: v for k, v in KEEP_COLS.items() if k in gsod_raw.columns})

# Filter to framework start year
gsod = gsod[gsod['year'] >= FRAMEWORK_START_YEAR].copy()
gsod = gsod.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {gsod.shape}")
print(f"Years: {gsod['year'].min()} — {gsod['year'].max()}")
print(f"Countries: {gsod['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (gsod.isnull().sum() / len(gsod) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Shape: (7216, 16)
Years: 1990 — 2025
Countries: 203

Missing values (%):
country_code                    15.0
idea_electoral_participation     3.4
idea_local_democracy             0.3
idea_civic_engagement            0.2
dtype: float64


In [5]:
# Check rows with missing country_code
missing_iso = gsod[gsod['country_code'].isnull()]['country_name'].unique()
print(f"Countries with missing ISO3 code ({len(missing_iso)}):")
print(missing_iso)

Countries with missing ISO3 code (29):
<StringArray>
[               'ASEAN',               'Africa',        'African Union',
             'Americas', 'Asia and the Pacific',            'Caribbean',
       'Central Africa',      'Central America',         'Central Asia',
       'Central Europe',          'East Africa',            'East Asia',
       'Eastern Europe',               'Europe',       'European Union',
         'North Africa', 'North/Western Europe',     'Northern America',
                  'OAS',                 'OECD',              'Oceania',
        'South America',           'South Asia',      'South-East Asia',
      'Southern Africa',      'Southern Europe',          'West Africa',
         'Western Asia',                'World']
Length: 29, dtype: str


In [6]:
# Drop regional aggregates — keep only rows with valid ISO3 country codes
gsod = gsod[gsod['country_code'].notna()].copy()
gsod = gsod.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape after dropping aggregates: {gsod.shape}")
print(f"Countries: {gsod['country_name'].nunique()}")

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "idea_gsod_clean.csv")
gsod.to_csv(output_path, index=False)
print(f"Written: {output_path}")

# Derive metadata — no hardcoding
latest_year = str(int(gsod['year'].max()))

# Update download log
update_entry(
    "IDEA_PARTIP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="idea_gsod_clean.csv",
    latest_available_version=f"v{gsod_version}",
    notes=f"IDEA Global State of Democracy indices. Manual CSV download — pipeline auto-detects latest version in Downloads. "
          f"13 category/subcomponent estimates covering participation, representation, rights, rule of law. "
          f"Coverage: 1990-{latest_year}, ~174 countries."
)
print_entry("IDEA_PARTIP")

Shape after dropping aggregates: (6137, 16)
Countries: 174
Written: C:\Users\mjbou\governance-framework\data\processed\idea_gsod_clean.csv
[download_log] Updated entry for IDEA_PARTIP
  source_id: IDEA_PARTIP
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2025
  local_filename: idea_gsod_clean.csv
  latest_available_version: v10
  no_update_reason: nan
  notes: IDEA Global State of Democracy indices. Manual CSV download — pipeline auto-detects latest version in Downloads. 13 category/subcomponent estimates covering participation, representation, rights, rule of law. Coverage: 1990-2025, ~174 countries.
